# Day 7: FastAPI Credit-Scoring Service

**Goal:** take the Day-5 MLP credit model out of the notebook and put it behind a web API.

By the end you can: serialize the trained model with `joblib`, wrap it in a FastAPI `/predict` endpoint, validate the incoming JSON with Pydantic, and call the running API from Python with `requests` (HTTP 200 for a good request, HTTP 422 for a bad one).

**Connection to Day 5:** the model and the data are exactly the ones you built in `Day5_MLP_Credit_Scoring.ipynb`. Day 7 does not change the model. It changes how the outside world talks to it.


### Setup: install the API libraries

The next cell installs FastAPI, uvicorn (the server that runs it), Pydantic (input validation), `nest_asyncio` (lets the server run inside Colab), `requests` (to call the API), and `joblib` (to save/load the model). Run it once; it takes about 20 seconds. `# [FULL]`

In [ ]:
!pip -q install fastapi==0.110.0 uvicorn==0.27.1 pydantic==2.6.1 nest_asyncio requests joblib
print('FastAPI stack installed.')

## Step 1: Rebuild the Day-5 model (verbatim)

The next two cells are copied unchanged from `Day5_MLP_Credit_Scoring.ipynb` (commit `0a7ad3a9`). They generate the 500-row synthetic credit dataset and import the libraries the model needs. We re-run them here so the notebook is self-contained.

In [ ]:
# source: Day5_MLP_Credit_Scoring.ipynb @ 0a7ad3a9
# Cell 0: Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)

NAVY  = '#1e3a5f'
AMBER = '#d97706'
SLATE = '#64748b'


In [ ]:
# source: Day5_MLP_Credit_Scoring.ipynb @ 0a7ad3a9
# Cell 1: Generate synthetic credit dataset (n=500)
n = 500

income           = np.random.uniform(20_000, 120_000, n)
debt_ratio       = np.random.uniform(0.1, 0.9, n)
credit_score     = np.random.uniform(300, 850, n)
employment_years = np.random.uniform(0, 20, n)
loan_amount      = np.random.uniform(5_000, 50_000, n)

# Approval rule: score > 650 AND debt < 0.45 AND income > 45k, plus 8% noise
base  = ((credit_score > 650) & (debt_ratio < 0.45) & (income > 45_000)).astype(int)
noise = np.random.binomial(1, 0.08, n)
y     = np.where(noise, 1 - base, base)

feature_names = ['income', 'debt_ratio', 'credit_score', 'employment_years', 'loan_amount']
X = np.column_stack([income, debt_ratio, credit_score, employment_years, loan_amount])

df = pd.DataFrame(X, columns=feature_names)
df['approved'] = y


### The five features, in plain finance terms

| Feature | What it means | Plausible range |
|---|---|---|
| `income` | yearly salary, the capacity to repay | 20,000 to 120,000 |
| `debt_ratio` | fraction of income already going to other debt | 0.1 to 0.9 |
| `credit_score` | track record of paying on time | 300 (poor) to 850 (excellent) |
| `employment_years` | job tenure, a stability signal | 0 to 20 |
| `loan_amount` | size of the new loan being requested | 5,000 to 50,000 |

Hold on to these ranges. They are what the Pydantic validator will police later: a request with `income = -3000` or `credit_score = 9000` is garbage, and the API should say so instead of feeding it to the model.

In [ ]:
# [FULL] Split, scale, and train -- same recipe as Day 5
# Split FIRST, then fit the scaler ONLY on the training rows (no leakage).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Same MLP architecture as Day 5: input(5) -> hidden(64) -> hidden(32) -> output(2)
clf = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
clf.fit(X_train_sc, y_train)

print(f'Trained. Test accuracy: {accuracy_score(y_test, clf.predict(X_test_sc)):.3f}')

## Step 2: Serialize the model with joblib

An API server is a separate program from your training notebook. It cannot "see" the `clf` variable. So we write the trained model (and the scaler, because the scaler is part of the model: it must standardize incoming requests with the SAME means and stds) to disk with `joblib.dump`, and the API will `joblib.load` them back.

Think of it like a recipe card. Training is cooking the dish once and tasting it. `joblib.dump` writes down the exact recipe so a different cook (the API server) can reproduce it without starting from raw ingredients.

In [ ]:
# [FULL] joblib dump / load round-trip
import joblib

# Bundle the model AND the scaler together -- they travel as one unit.
joblib.dump({'model': clf, 'scaler': scaler, 'features': feature_names}, 'credit_model.joblib')
print('Saved credit_model.joblib')

# Load it back (this is what the API will do at startup) and check it still works.
bundle = joblib.load('credit_model.joblib')
loaded_model, loaded_scaler = bundle['model'], bundle['scaler']

sample = np.array([[60_000, 0.30, 720, 5, 15_000]])  # a plausibly-good applicant
sample_sc = loaded_scaler.transform(sample)
print('Reloaded model approval probability for the sample:',
      round(float(loaded_model.predict_proba(sample_sc)[0, 1]), 3))
print('Same number you would get from the original clf -- the round-trip is lossless.')

## Step 3: What is an API? (read this before any code)

**An API is a waiter. The model is the kitchen.** You (the client) hand the waiter your order (an HTTP request: "here are five numbers for an applicant"). The waiter carries it to the kitchen (your loaded model), the kitchen cooks (`predict_proba`), and the waiter brings back the dish (a prediction, as JSON: `{"approval_probability": 0.87, "decision": "approve"}`).

You never walk into the kitchen yourself. You do not need to know Python, scikit-learn, or how an MLP works. You just need the menu: which URL to POST to, and what shape the JSON should be. That is the whole idea of a REST API.

Three things to keep straight:
- **Endpoint**: a URL the server answers, like `/predict` or `/health`. One waiter station.
- **HTTP method**: `GET` means "show me something" (read), `POST` means "here is data, do something with it" (our predictions are `POST`).
- **Status code**: a number the waiter says back. `200` = here is your dish. `422` = your order does not make sense (you asked for a steak with a negative weight). `500` = the kitchen caught fire.

### Instructor demo: the smallest possible API

Before we wrap the credit model, here is a toy API with one endpoint, `/add`, that returns `a + b`. We run it INSIDE Colab (Colab has no public port, but `127.0.0.1` works for talking to ourselves), then call it with `requests`. This is the same pattern we will use for `/predict`, just with arithmetic instead of a model.

In [ ]:
# [FULL]
# INSTRUCTOR DEMO: a minimal FastAPI app with one /add endpoint, run in-Colab.
import time
from fastapi import FastAPI
import nest_asyncio
import uvicorn
from threading import Thread
import requests

demo_app = FastAPI()

@demo_app.get('/add')
def add(a: float, b: float):
    return {'a': a, 'b': b, 'sum': a + b}

# Colab trick: nest_asyncio lets uvicorn run inside the already-running notebook event loop.
nest_asyncio.apply()
Thread(target=lambda: uvicorn.run(demo_app, host='127.0.0.1', port=8000), daemon=True).start()
time.sleep(2)  # give the server a moment to come up

resp = requests.get('http://127.0.0.1:8000/add', params={'a': 2, 'b': 3})
print('Status code:', resp.status_code)   # expect 200
print('Response JSON:', resp.json())       # expect {'a': 2.0, 'b': 3.0, 'sum': 5.0}

That is a working web service. It took about ten lines. `/predict` will be the same shape: define an app, define an endpoint, run it on `127.0.0.1` in a background thread, call it with `requests`. The only new ingredient is loading the model once at module scope and validating the input.

(FastAPI also auto-generates interactive docs at `/docs`. In Colab that page is awkward to click, so we will test everything from Python with `requests` instead. You can mention `/docs` exists; we will not depend on it.)

## CHECKPOINT 1: What shape is the `/predict` request JSON?

Before we build the endpoint, predict (discuss with your neighbour): the client wants a prediction for one applicant. What should the JSON body look like? Which keys, what types?

Write your guess here, then read on:

```json
{ ... your guess ... }
```

## Step 4: The wrong way first -- no input validation

Here is a `/predict` endpoint that takes whatever numbers it is handed and feeds them straight to the model. No checks. Watch what happens when we send an applicant with a NEGATIVE income and a credit score of 9000 (impossible -- the scale stops at 850).

In [ ]:
# [MIN]
# WRONG WAY: no input validation -- garbage in, confident garbage out.
from fastapi import FastAPI

bad_app = FastAPI()
_bundle = joblib.load('credit_model.joblib')   # at least we load once, not per-request
_model, _scaler = _bundle['model'], _bundle['scaler']

@bad_app.post('/predict')
def predict_no_checks(payload: dict):
    # payload is just a plain dict -- FastAPI does not check anything.
    row = [[payload['income'], payload['debt_ratio'], payload['credit_score'],
            payload['employment_years'], payload['loan_amount']]]
    row_sc = _scaler.transform(row)
    prob = float(_model.predict_proba(row_sc)[0, 1])
    return {'approval_probability': round(prob, 3)}

# Simulate the call WITHOUT starting a server -- just run the function with junk input.
junk = {'income': -3000, 'debt_ratio': 5.0, 'credit_score': 9000,
        'employment_years': -1, 'loan_amount': 999_999}
print('Junk applicant ->', predict_no_checks(junk))
print()
print('The model returned a confident-looking probability for an applicant who cannot exist.')
print('Negative income. Credit score off the scale. The model has no idea -- it just extrapolates.')
print('In production this is how a credit API approves a loan for nonsense data.')

### The fix: a Pydantic model that rejects impossible applicants

Pydantic lets us declare the shape AND the bounds of the request. `income` must be a number greater than 0. `credit_score` must be between 300 and 850. `debt_ratio` between 0 and 1. If a request violates any of these, FastAPI never calls our function -- it returns HTTP **422 Unprocessable Entity** with a message saying exactly which field was wrong. The model only ever sees applicants that are at least physically possible.

In [ ]:
# [FULL] The validated request schema
from pydantic import BaseModel, Field

class CreditApplication(BaseModel):
    income: float           = Field(gt=0,            description='Yearly income, must be positive')
    debt_ratio: float       = Field(ge=0.0, le=1.0,  description='Fraction of income committed to debt')
    credit_score: float     = Field(ge=300, le=850,  description='FICO-style score, 300 to 850')
    employment_years: float = Field(ge=0, le=60,     description='Years employed')
    loan_amount: float      = Field(gt=0,            description='Requested loan size, positive')

# Quick local sanity check (no server needed yet):
ok = CreditApplication(income=60_000, debt_ratio=0.30, credit_score=720,
                       employment_years=5, loan_amount=15_000)
print('Valid application parsed:', ok.model_dump())

try:
    CreditApplication(income=-3000, debt_ratio=5.0, credit_score=9000,
                      employment_years=-1, loan_amount=999_999)
except Exception as e:
    print()
    print('Pydantic rejected the junk applicant. First few errors:')
    for err in e.errors()[:3]:
        print(f"  field={err['loc']}  problem={err['msg']}")

## CHECKPOINT 2: What HTTP status does a negative-income request return?

We are about to start the real API and send it a request with `income = -5000`. Predict the status code that comes back. (Discuss with your neighbour: who decides it -- our code, or FastAPI?)

Your prediction: ________

## Step 5: The real credit-scoring API

Now we put it together: load the model bundle ONCE at module scope (a comment marks this -- it is the single most important performance rule for ML APIs: loading a model can take hundreds of milliseconds, and you do NOT want that on every request), define `/health` and `/predict`, validate with `CreditApplication`, and run it on `127.0.0.1` in a background thread so we can call it from later cells.

In [ ]:
# [FULL] The credit-scoring API -- model loaded once, input validated, /predict returns JSON.
import time
from threading import Thread
from fastapi import FastAPI
import nest_asyncio
import uvicorn
import requests

app = FastAPI(title='Credit Scoring API')

# --- Model loaded ONCE here, at module scope -- NOT inside the endpoint. ---
# Loading on every request would add ~hundreds of ms each time. Load once, reuse forever.
_bundle = joblib.load('credit_model.joblib')
MODEL, SCALER, FEATURES = _bundle['model'], _bundle['scaler'], _bundle['features']

@app.get('/health')
def health():
    return {'status': 'ok', 'model_loaded': MODEL is not None}

@app.post('/predict')
def predict(application: CreditApplication):
    # 'application' is already validated by Pydantic. If we got here, the numbers are sane.
    row = [[application.income, application.debt_ratio, application.credit_score,
            application.employment_years, application.loan_amount]]
    row_sc = SCALER.transform(row)
    prob = float(MODEL.predict_proba(row_sc)[0, 1])
    return {
        'approval_probability': round(prob, 4),
        'decision': 'approve' if prob >= 0.5 else 'decline',
    }

nest_asyncio.apply()
Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8001), daemon=True).start()
time.sleep(2)
print('Credit Scoring API running on http://127.0.0.1:8001')
print('Health check:', requests.get('http://127.0.0.1:8001/health').json())

In [ ]:
# [FULL] Test 1: a valid request -- expect HTTP 200 and a JSON probability.
good_applicant = {
    'income': 72_000,
    'debt_ratio': 0.28,
    'credit_score': 740,
    'employment_years': 6,
    'loan_amount': 18_000,
}
r = requests.post('http://127.0.0.1:8001/predict', json=good_applicant)
print('Status code:', r.status_code)   # expect 200
print('Response JSON:', r.json())       # expect {'approval_probability': 0.xx, 'decision': '...'}
assert r.status_code == 200
assert 'approval_probability' in r.json()
print('OK -- valid request returned 200 with a probability.')

In [ ]:
# [FULL] Test 2: an invalid request (negative income) -- expect HTTP 422.
bad_applicant = {
    'income': -5000,          # impossible
    'debt_ratio': 0.28,
    'credit_score': 740,
    'employment_years': 6,
    'loan_amount': 18_000,
}
r = requests.post('http://127.0.0.1:8001/predict', json=bad_applicant)
print('Status code:', r.status_code)   # expect 422
print('Response JSON:', r.json())       # FastAPI lists the offending field
assert r.status_code == 422
print('OK -- invalid request returned 422 before the model was ever called.')
print()
# One more: credit_score off the scale.
r2 = requests.post('http://127.0.0.1:8001/predict', json={**good_applicant, 'credit_score': 9000})
print('credit_score=9000 ->', r2.status_code, r2.json())

## What you just built

- A trained model serialized with `joblib` (model + scaler bundled together).
- A FastAPI service that loads that model ONCE at startup, not per request.
- A Pydantic schema that rejects impossible applicants with HTTP 422.
- A working `/predict` endpoint that returns a JSON probability, tested from Python with `requests`: 200 for a good request, 422 for a bad one.

The model never changed. We just gave it a front door that any program, in any language, can knock on.


**Homework (in the SOLVED notebook):** wrap `app` in a `main.py`, add a `requirements.txt`, and deploy to Hugging Face Spaces or Render for a real public URL.